In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/soda-challenge/soda_dataset/sample_submission.csv
/kaggle/input/competitions/soda-challenge/soda_dataset/train.csv
/kaggle/input/competitions/soda-challenge/soda_dataset/test.csv
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/208.jpg
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/641.png
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/473.jpg
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/491.png
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/709.png
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/537.jpg
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/45.jpg
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/654.jpg
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/480.png
/kaggle/input/competitions/soda-challenge/soda_dataset/images/test/89.jpg
/kaggle/input/competitions/soda-challenge/soda

In [2]:
import numpy as np
import pandas as pd
import random
import os
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils import class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE = '/kaggle/input/competitions/soda-challenge/soda_dataset/'
TRAIN_PATH = BASE + 'images/train/'
TEST_PATH = BASE + 'images/test/'

IMG_SIZE = 384
BATCH_SIZE = 12
EPOCHS_STAGE1 = 8
EPOCHS_STAGE2 = 5
N_FOLDS = 5
NUM_CLASSES = 13

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
train['label'] = train['label'].astype(str)

generator_classes = [str(i) for i in sorted(train['label'].astype(int).unique())]
train['label_idx'] = train['label'].astype(int)

def get_filename(img_id, path):
   for ext in ['.jpg', '.png', '.jpeg']:
       if os.path.exists(path + str(img_id) + ext):
           return str(img_id) + ext
   return str(img_id) + '.jpg'

train['filename'] = train['id'].apply(lambda x: get_filename(x, TRAIN_PATH))
test['filename'] = test['id'].apply(lambda x: get_filename(x, TEST_PATH))

# CLASS WEIGHTS - الإصلاح
class_weights_arr = class_weight.compute_class_weight(
   class_weight='balanced',
   classes=np.unique(train['label']),
   y=train['label']
)
class_weights_dict = {i: class_weights_arr[i] for i in range(len(class_weights_arr))}

def build_model():
   base = EfficientNetV2S(
       weights='imagenet',
       include_top=False,
       input_shape=(IMG_SIZE, IMG_SIZE, 3)
   )
   for layer in base.layers:
       layer.trainable = False

   x = base.output
   x = GlobalAveragePooling2D()(x)
   x = BatchNormalization()(x)
   x = Dropout(0.5)(x)
   x = Dense(512, activation='swish')(x)
   x = Dropout(0.4)(x)
   output = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

   model = Model(inputs=base.input, outputs=output)
   return model, base

train_datagen = ImageDataGenerator(
   preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input,
   rotation_range=40,
   width_shift_range=0.2,
   height_shift_range=0.2,
   zoom_range=0.2,
   horizontal_flip=True,
   vertical_flip=True,
   brightness_range=[0.7, 1.3],
   shear_range=0.15,
   fill_mode='reflect'
)

valid_datagen = ImageDataGenerator(
   preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input
)

oof_preds = np.zeros((len(train), NUM_CLASSES))
test_preds = np.zeros((len(test), NUM_CLASSES))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['label_idx'])):
   print(f"\n{'='*40}")
   print(f"🚀 FOLD {fold+1}/{N_FOLDS}")
   print(f"{'='*40}\n")

   train_df = train.iloc[train_idx]
   val_df = train.iloc[val_idx]

   train_generator = train_datagen.flow_from_dataframe(
       dataframe=train_df, directory=TRAIN_PATH,
       x_col='filename', y_col='label',
       target_size=(IMG_SIZE, IMG_SIZE),
       batch_size=BATCH_SIZE, class_mode='categorical',
       classes=generator_classes, shuffle=True, seed=SEED
   )

   val_generator = valid_datagen.flow_from_dataframe(
       dataframe=val_df, directory=TRAIN_PATH,
       x_col='filename', y_col='label',
       target_size=(IMG_SIZE, IMG_SIZE),
       batch_size=BATCH_SIZE, class_mode='categorical',
       classes=generator_classes, shuffle=False
   )

   model, base_model = build_model()

   model.compile(
       optimizer=Adam(learning_rate=1e-4),
       loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
       metrics=['accuracy']
   )

   stage1_callbacks = [
       EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
       ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
       ModelCheckpoint(f'best_fold_{fold}.keras', monitor='val_loss', save_best_only=True)
   ]

   print("🚀 STAGE 1...")
   model.fit(
       train_generator, validation_data=val_generator,
       epochs=EPOCHS_STAGE1, callbacks=stage1_callbacks,
       class_weight=class_weights_dict, verbose=1
   )

   print("\n🔥 STAGE 2 Fine-Tuning...")
   for layer in base_model.layers[:-100]:
       layer.trainable = False
   for layer in base_model.layers[-100:]:
       layer.trainable = True

   model.compile(
       optimizer=Adam(learning_rate=1e-5),
       loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
       metrics=['accuracy']
   )

   stage2_callbacks = [
       EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
       ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1, min_lr=1e-7),
       ModelCheckpoint(f'best_fold_{fold}.keras', monitor='val_loss', save_best_only=True)
   ]

   model.fit(
       train_generator, validation_data=val_generator,
       epochs=EPOCHS_STAGE2, callbacks=stage2_callbacks,
       class_weight=class_weights_dict, verbose=1
   )

   val_generator.reset()
   preds = model.predict(val_generator, verbose=1)
   oof_preds[val_idx] = preds

   fold_f1 = f1_score(
       train.iloc[val_idx]['label_idx'],
       np.argmax(preds, axis=1),
       average='macro'
   )
   print(f"\n✅ Fold {fold+1} F1: {fold_f1:.4f}")

   print("\n🎯 TTA PREDICTIONS...")
   fold_test_preds = np.zeros((len(test), NUM_CLASSES))

   tta_configs = [
       {'rotation_range': 15, 'zoom_range': 0.1, 'horizontal_flip': True},
       {'rotation_range': 25, 'zoom_range': 0.15, 'horizontal_flip': True},
       {'rotation_range': 10, 'zoom_range': 0.05, 'horizontal_flip': False},
       {'rotation_range': 30, 'zoom_range': 0.2, 'horizontal_flip': True},
       {'rotation_range': 20, 'zoom_range': 0.1, 'horizontal_flip': True},
   ]

   for i, config in enumerate(tta_configs):
       print(f"TTA Round {i+1}/5...")
       tta_gen = ImageDataGenerator(
           preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input,
           **config
       )
       test_generator = tta_gen.flow_from_dataframe(
           dataframe=test, directory=TEST_PATH,
           x_col='filename', y_col=None,
           target_size=(IMG_SIZE, IMG_SIZE),
           batch_size=BATCH_SIZE, class_mode=None, shuffle=False
       )
       fold_test_preds += model.predict(test_generator, verbose=0)

   fold_test_preds /= 5
   test_preds += fold_test_preds

test_preds /= N_FOLDS

oof_labels = np.argmax(oof_preds, axis=1)
true_labels = train['label_idx'].values

final_f1 = f1_score(true_labels, oof_labels, average='macro')
print(f"\n🎯 FINAL OOF F1 SCORE: {final_f1:.5f}")
print(classification_report(true_labels, oof_labels))

pred_labels = np.argmax(test_preds, axis=1)

submission = pd.DataFrame({
   'id': test['id'],
   'label': pred_labels
})
submission.to_csv('submission.csv', index=False)
print("\n✅ SUBMISSION SAVED!")
print(submission.head(10))
print("\nتوزيع التنبؤات:")
print(submission['label'].value_counts())


2026-05-29 10:54:07.419398: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780052047.611272      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780052047.664692      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780052048.120030      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780052048.120070      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780052048.120073      23 computation_placer.cc:177] computation placer alr


🚀 FOLD 1/5

Found 1463 validated image filenames belonging to 13 classes.
Found 366 validated image filenames belonging to 13 classes.


I0000 00:00:1780052075.608153      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780052075.614477      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
🚀 STAGE 1...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


...

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 312s 2s/step - accuracy: 0.0850 - loss: 3.8890 - val_accuracy: 0.2493 - val_loss: 2.3836 - learning_rate: 1.0000e-04
Epoch 2/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.1716 - loss: 3.0935 - val_accuracy: 0.3781 - val_loss: 2.1654 - learning_rate: 1.0000e-04
Epoch 3/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.2168 - loss: 2.8846 - val_accuracy: 0.4137 - val_loss: 2.0149 - learning_rate: 1.0000e-04
Epoch 4/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 0.2278 - loss: 2.7184 - val_accuracy: 0.4438 - val_loss: 1.9307 - learning_rate: 1.0000e-04
Epoch 5/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.2663 - loss: 2.6307 - val_accuracy: 0.4466 - val_loss: 1.9040 - learning_rate: 1.0000e-04
Epoch 6/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.2725 - loss: 2.5403 - val_accuracy: 0.4795 - val_loss: 1.8725 - learning_rate: 1.0000e-04
Epoch 7/8
122/122 ━━━━━━━━━━━━━━━━━━━━ 203s 2s/step - accuracy: 

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


TTA Round 2/5...
Found 784 validated image filenames.
TTA Round 3/5...
Found 784 validated image filenames.
TTA Round 4/5...
Found 784 validated image filenames.
TTA Round 5/5...
Found 784 validated image filenames.

🎯 FINAL OOF F1 SCORE: 0.03593
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.02      0.01      0.01       141
           2       0.04      0.04      0.04       141
           3       0.09      0.09      0.09       141
           4       0.06      0.06      0.06       141
           5       0.02      0.03      0.02       141
           6       0.02      0.02      0.02       140
           7       0.09      0.10      0.09       141
           8       0.04      0.04      0.04       140
           9       0.01      0.01      0.01       141
          10       0.01      0.01      0.01       140
          11       0.05      0.05      0.05       141
          12       0.06      0.04      0.05       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_